In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from utils.const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import get_data, setup_device

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

loader_train = data_module.train_dataloader()
loader_val = data_module.val_dataloader()
loader_test = data_module.test_dataloader()

data_train = get_data(loader_train)
data_val = get_data(loader_val)
data_test = get_data(loader_test)

data = torch.cat([data_train, data_val, data_test], dim=0)
print(data.shape)

num_samples = data.shape[0]


In [ ]:
from utils.anatomical_validity_checker import AnatomicalValidityChecker

for data in [data_train, data_val, data_test]:
    num_valid = 0

    for slice in data:
        AV = AnatomicalValidityChecker(slice)
        if AV.count_violations() == 0:
            num_valid += 1

    print(f"% AV: {num_valid / len(data)}")